In [ ]:

!pip install transformers torch scikit-learn fairlearn aif360 pandas matplotlib seaborn datasets accelerate -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    average_precision_score
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from torch.utils.data import Dataset

# Set seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
DATA_PATH = 'jigsaw-unintended-bias-train.csv'

print("Loading dataset...")
df = pd.read_csv(DATA_PATH, usecols=['comment_text', 'toxic', 'black', 'white', 'muslim', 'jewish', 'identity_attack'])

print(f"Full dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nToxic score distribution (sample):")
print(df['toxic'].describe())

In [ ]:
df['label'] = (df['toxic'] >= 0.5).astype(int)

df = df.dropna(subset=['comment_text'])
df['comment_text'] = df['comment_text'].astype(str)

print(f"Class distribution after binarization:")
print(df['label'].value_counts())
print(f"Toxic rate: {df['label'].mean():.3f}")

In [ ]:
df_sample, _ = train_test_split(
    df, train_size=120000, stratify=df['label'], random_state=SEED
)

df_train, df_eval = train_test_split(
    df_sample, test_size=20000, stratify=df_sample['label'], random_state=SEED
)

df_train = df_train.reset_index(drop=True)
df_eval = df_eval.reset_index(drop=True)

print(f"Training set: {len(df_train)} rows")
print(f"  Toxic rate: {df_train['label'].mean():.4f}")
print(f"\nEvaluation set: {len(df_eval)} rows")
print(f"  Toxic rate: {df_eval['label'].mean():.4f}")

df_train.to_csv('train_subset.csv', index=False)
df_eval.to_csv('eval_subset.csv', index=False)
print("\nSaved train_subset.csv and eval_subset.csv")

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ToxicityDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = ToxicityDataset(
    df_train['comment_text'].tolist(),
    df_train['label'].tolist(),
    tokenizer
)

eval_dataset = ToxicityDataset(
    df_eval['comment_text'].tolist(),
    df_eval['label'].tolist(),
    tokenizer
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Eval dataset size: {len(eval_dataset)}")

sample = train_dataset[0]
print(f"Sample keys: {sample.keys()}")
print(f"input_ids shape: {sample['input_ids'].shape}")

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {'f1_macro': f1, 'accuracy': acc}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir='./results_part1',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=200,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
trainer.save_model('./model_checkpoint_part1')
tokenizer.save_pretrained('./model_checkpoint_part1')
print("Model saved to ./model_checkpoint_part1")

In [ ]:
import torch.nn.functional as F

predictions_output = trainer.predict(eval_dataset)
logits = predictions_output.predictions
probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
prob_toxic = probs[:, 1]  # probability of class 1 (toxic)
true_labels = df_eval['label'].values

np.save('eval_probs_part1.npy', prob_toxic)
np.save('eval_labels.npy', true_labels)

print("Probabilities computed and saved.")
print(f"Probability range: [{prob_toxic.min():.4f}, {prob_toxic.max():.4f}]")
print(f"Mean predicted probability: {prob_toxic.mean():.4f}")

In [ ]:

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
threshold_results = []

for t in thresholds:
    preds = (prob_toxic >= t).astype(int)
    f1_mac = f1_score(true_labels, preds, average='macro')
    f1_tox = f1_score(true_labels, preds, pos_label=1)
    acc = accuracy_score(true_labels, preds)
    cm = confusion_matrix(true_labels, preds)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    threshold_results.append({
        'Threshold': t, 'Accuracy': acc, 'F1 Macro': f1_mac,
        'F1 Toxic': f1_tox, 'FPR': fpr, 'FNR': fnr
    })

threshold_df = pd.DataFrame(threshold_results)
print("Threshold Analysis:")
print(threshold_df.to_string(index=False))

In [ ]:
CHOSEN_THRESHOLD = 0.5
final_preds = (prob_toxic >= CHOSEN_THRESHOLD).astype(int)

acc = accuracy_score(true_labels, final_preds)
f1_mac = f1_score(true_labels, final_preds, average='macro')
auc = roc_auc_score(true_labels, prob_toxic)
cm = confusion_matrix(true_labels, final_preds)

print("=" * 50)
print(f"FINAL METRICS at threshold={CHOSEN_THRESHOLD}")
print("=" * 50)
print(f"Accuracy:      {acc:.4f}")
print(f"F1 (Macro):    {f1_mac:.4f}")
print(f"AUC-ROC:       {auc:.4f}")
print(f"\nConfusion Matrix:")
print(cm)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Non-Toxic', 'Toxic'],
            yticklabels=['Non-Toxic', 'Toxic'])
ax.set_title(f'Confusion Matrix (threshold={CHOSEN_THRESHOLD})')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('part1_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
fpr_arr, tpr_arr, _ = roc_curve(true_labels, prob_toxic)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr_arr, tpr_arr, 'b-', lw=2, label=f'ROC (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

precision_arr, recall_arr, _ = precision_recall_curve(true_labels, prob_toxic)
ap = average_precision_score(true_labels, prob_toxic)
axes[1].plot(recall_arr, precision_arr, 'r-', lw=2, label=f'PR Curve (AP = {ap:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('part1_roc_pr_curves.png', dpi=150)
plt.show()

print(f"AUC-ROC: {auc:.4f}")
print(f"Average Precision (PR AUC): {ap:.4f}")

## 1.5 Threshold Justification

**Chosen threshold: 0.5**

From the threshold sweep:
- Lower thresholds (0.3–0.4) improve recall of toxic content but significantly increase False Positive Rate — innocent users are incorrectly flagged more often.
- Higher thresholds (0.6–0.7) reduce false positives but miss a larger fraction of genuinely toxic comments (higher FNR), allowing harmful content to remain on the platform.

**Platform priorities implication:**

The threshold 0.5 reflects a neutral starting point where neither error direction is systematically prioritized. For this assignment, 0.5 is chosen as the baseline because:
1. It is the most interpretable and standard default
2. The bias audit in Part 2 specifically examines FPR disparities — using 0.5 gives a clean, untuned baseline to compare against mitigated models
3. The Guardrail Pipeline (Part 5) will add a human review band (0.4–0.6) to handle uncertain cases, effectively making the hard threshold less critical

A platform prioritizing user experience and minimizing wrongful removals should use a higher threshold (0.6–0.7). A platform where toxic content causes significant harm (e.g., harassment campaigns) may prefer a lower threshold (0.3–0.4) despite more false positives. There is no universally correct answer — this trade-off reflects the platform's values and legal exposure.